# Federated Learning Server
## Central Aggregation Server for Distributed Model Training

This notebook implements the **FL Server** for a Federated Learning simulation.

---

### Server Responsibilities

| Role | Description |
|------|-------------|
| **Global Model Management** | Initialize, store, and distribute the global model |
| **Weight Collection** | Receive trained weights from each client |
| **Federated Aggregation** | Perform FedAvg to combine client weights |
| **Model Evaluation** | Evaluate aggregated model on test data |

---

## Quick Start Guide (Google Colab)

**Prerequisites:**
1. Create a free ngrok account: https://dashboard.ngrok.com/signup
2. Copy your authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken


> **Note:** For this workshop, the authtokens for all participant groups are available in the GitHub repository.



**Running the Server:**
1. Paste your ngrok authtoken in the configuration cell
2. Run all cells until the server is running
3. **Copy the ngrok public URL** (e.g., `https://xxxx.ngrok-free.app`)
4. Open `client_1.ipynb` and `client_2.ipynb` in separate tabs
5. Paste the server URL in each client's configuration
6. Run both clients to start federated training



# 1. Environment Setup

Before running the FL server, we need to set up the computing environment with all required dependencies.

### 1.1 Install Dependencies
Install required Python packages for deep learning, web server, and network tunneling.

In [ ]:
# Install dependencies
!pip install flask efficientnet_pytorch pyngrok -q

  Preparing metadata (setup.py) ... done


### 1.2 Import Libraries
Import all necessary libraries for the FL server implementation.

In [ ]:
import os
import io
import copy
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime
from flask import Flask, request, jsonify, send_file
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from efficientnet_pytorch import EfficientNet
import threading
import gzip
import base64
from pyngrok import ngrok, conf
print('Libraries imported!')

Libraries imported!


# 2. Data Setup

The server requires **test data** for evaluating the global model after each aggregation round. In this simulation, we download the test dataset from a GitHub repository.

In [ ]:
# ========================================
# DOWNLOAD DATASET FROM GITHUB
# ========================================

# GitHub repository
GITHUB_REPO_URL = 'https://github.com/YARSIAICenter/faradisa-workshop-2026'
DATA_FOLDER = 'day-3/data/test'  # Server only need test data
LOCAL_DIR = 'workshop-data'

# Download only folder data
if not os.path.exists(LOCAL_DIR):
    print('Downloading test dataset...')
    !git clone --filter=blob:none --sparse {GITHUB_REPO_URL} {LOCAL_DIR}
    %cd {LOCAL_DIR}
    !git sparse-checkout set {DATA_FOLDER}
    %cd ..
    print('Download complete!')
else:
    print(f'Data available in folder {LOCAL_DIR}/')

# Set BASE_DIR ke folder yang berisi data
BASE_DIR = os.path.join(LOCAL_DIR, 'day-3')
print(f'Data directory: {BASE_DIR}/')
print(f'Test data: {os.path.join(BASE_DIR, "data/test")}')

Cloning into 'workshop-data'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 43 (delta 5), reused 34 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 42.09 KiB | 14.03 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/workshop-data
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (251/251), done.
remote: Total 251 (delta 0), reused 251 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (251/251), 106.49 MiB | 18.29 MiB/s, done.
Updating files: 100% (251/251), done.
/content
Download complete!
Data directory: workshop-data/day-3/
Test data: workshop-data/day-3/data/test


# 3. Network Configuration

To enable communication between the server and clients running on different machines (or Colab instances), we use **ngrok** to create a public URL tunnel.

In a real-world deployment, you would:
- Host the server on a machine with a public IP address
- Configure firewall rules to allow incoming connections
- Use HTTPS with proper SSL certificates

### 3.1 Create Public URL Tunnel

**1. Setup Ngrok:**
1. Sign up for free at: https://dashboard.ngrok.com/signup
2. Get your authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste it in the cell below

**2. Start ngrok tunnel to expose the local Flask server to the internet.**

In [ ]:
# ========================================
# 3.1 SETUP NGROK AUTHTOKEN
# ========================================
# Register at: https://dashboard.ngrok.com/signup
# Copy your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
# Note: For this workshop, the authtokens for all participant groups are available in the GitHub repository.

NGROK_AUTHTOKEN = '...' # Paste your ngrok authtoken here

# Validasi
print('='*60)
print('SETUP NGROK AUTHTOKEN')
print('='*60)
if 'PASTE' in NGROK_AUTHTOKEN or len(NGROK_AUTHTOKEN) < 20:
    print('='*60)
    print('ERROR: NGROK_AUTHTOKEN is empty!')
    print('')
    print('1. Register at: https://dashboard.ngrok.com/signup')
    print('2. Copy your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken')
    print('3. Paste it to the NGROK_AUTHTOKEN variable')
    print('='*60)
else:
    print('Authtoken OK!')

# ========================================
# 3.2 START NGROK TUNNEL
# ========================================

# Set authtoken
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Kill any existing ngrok processes
ngrok.kill()

# Server config
PORT = 5000

# Start ngrok tunnel
public_url = ngrok.connect(PORT).public_url
SERVER_URL = public_url

print('='*60)
print('NGROK TUNNEL CREATED!')
print('='*60)
print(f'\n>>> PUBLIC SERVER URL: {SERVER_URL} <<<\n')
print('copy this URL to the notebook client_1 dan client_2')
print('='*60)

SETUP NGROK AUTHTOKEN
Authtoken OK!
NGROK TUNNEL CREATED!

>>> PUBLIC SERVER URL: https://lily-snowplow-moodiness.ngrok-free.dev <<<

copy this URL to the notebook client_1 dan client_2


### 3.3 Server Configuration
Define model and file paths for the FL server.

In [ ]:
# Model config
N_CLASSES = ... # Insert the number of classes of your datasets
CLASS_NAMES = ['...', '...'] # Insert all of the classes name

# Set paths
MODEL_DIR = 'models'
GLOBAL_MODEL_PATH = os.path.join(MODEL_DIR, 'global_model.pth')
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.pth')
TEST_DATA_DIR = os.path.join(BASE_DIR, 'data/test')

os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Number of Classes: {N_CLASSES}')
print(f'Class Names: {CLASS_NAMES}')
print(f'Global Model Path: {GLOBAL_MODEL_PATH}')
print(f'Best Model Path: {BEST_MODEL_PATH}')
print(f'Test Data Path: {TEST_DATA_DIR}')

Number of Classes: 2
Class Names: ['Normal', 'Abnormal']
Global Model Path: models/global_model.pth
Best Model Path: models/best_model.pth
Test Data Path: workshop-data/day-3/data/test


# 4. Model Architecture

Define the neural network architecture. This must be **identical** across the server and all clients to ensure weight compatibility during aggregation.

**EfficientNet-B0 Model Definition**
Define the model architecture using EfficientNet-B0 as the backbone with a custom classification head.

In [ ]:
class EfficientNetB0(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.model = EfficientNet.from_pretrained('efficientnet-b0')
        self.num_ftrs = self.model._fc.in_features
        self.model._fc = nn.Linear(self.num_ftrs, n_classes)
        self.projector = nn.Sequential(
            nn.Linear(self.num_ftrs, self.num_ftrs),
            nn.Linear(self.num_ftrs, 1024)
        )

    def forward(self, x, project=False):
        features = self.model.extract_features(x)
        features = self.model._avg_pooling(features)
        features = features.flatten(start_dim=1)
        out = self.model._dropout(features)
        out = self.model._fc(out)
        return features, out

print('Model architecture defined!')

Model architecture defined!


# 5. Federated Learning Logic

This section implements the core FL components to initialize server state variables and implement the Federated Averaging algorithm.:
- **FL State Management**: Track training rounds, registered clients, and received weights
- **FedAvg Algorithm**: Weighted averaging of model parameters based on client data sizes

In [ ]:
# FL State
fl_state = {
    'current_round': 0,
    'registered_clients': {},
    'received_weights': {},
    'client_data_sizes': {},
    'is_aggregating': False,
    'expected_clients': ..., # Insert the  number of clients
    'best_bacc': 0.0,
    'best_round': -1,
    'evaluation_history': []
}

chunked_uploads = {}

def fedavg(weights_list, data_sizes):
    '''FedAvg: Weighted average berdasarkan jumlah data'''
    total = sum(data_sizes)
    w_avg = copy.deepcopy(weights_list[0])
    for key in w_avg.keys():
        w_avg[key] = w_avg[key] * (data_sizes[0] / total)
        for i in range(1, len(weights_list)):
            w_avg[key] += weights_list[i][key] * (data_sizes[i] / total)
    return w_avg

def save_global_model(state_dict):
    torch.save(state_dict, GLOBAL_MODEL_PATH)
    print(f'Global model saved to {GLOBAL_MODEL_PATH}')

print('FL state and helpers initialized!')

FL state and helpers initialized!


# 6. Model Evaluation

After each aggregation round, the server evaluates the new global model on the test dataset to track model performance.

**Test Dataset and Evaluation Function:**
Define the test dataset loader and evaluation metrics (accuracy, balanced accuracy).

In [ ]:
class TestDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.samples = []
        self.transform = transform
        csv_path = os.path.join(data_dir, 'labels.csv')
        images_dir = os.path.join(data_dir, 'images')
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            img_path = os.path.join(images_dir, row['filename'])
            if os.path.exists(img_path):
                self.samples.append((img_path, int(row['label'])))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, label

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def evaluate_model(state_dict):
    '''Evaluate global model on test data'''
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = EfficientNetB0(N_CLASSES).to(device)
    model.load_state_dict(state_dict)
    model.eval()

    test_dataset = TestDataset(TEST_DATA_DIR, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            _, logits = model(images.to(device))
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    acc = np.mean(all_preds == all_labels)

    # Balanced accuracy
    recall_per_class = []
    for c in range(N_CLASSES):
        mask = all_labels == c
        if np.sum(mask) > 0:
            recall_per_class.append(np.sum((all_preds == c) & mask) / np.sum(mask))
    bacc = np.mean(recall_per_class)

    return {'accuracy': acc, 'balanced_accuracy': bacc, 'test_size': len(test_dataset)}

print('Test dataset and evaluation ready!')

Test dataset and evaluation ready!


# 7. Global Model Initialization

Initialize the global model with pretrained ImageNet weights. This serves as the starting point for all clients in round 0.

**Initialize and Save Global Model**
Create the initial global model using pretrained EfficientNet-B0 weights.

In [ ]:
# Initialize global model from pretrained
print('Initializing global model from pretrained...')
model = EfficientNetB0(N_CLASSES)
save_global_model(model.state_dict())
print('Global model initialized!')

Initializing global model from pretrained...
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 106MB/s] 


Loaded pretrained weights for efficientnet-b0
Global model saved to models/global_model.pth
Global model initialized!


# 8. API Server Implementation

The FL server exposes REST API endpoints for client communication:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | GET | Server health check |
| `/register` | POST | Client registration |
| `/model/download` | GET | Download current global model |
| `/model/upload_chunk` | POST | Upload model weights (chunked) |
| `/model/upload_complete` | POST | Signal upload completion |
| `/status` | GET | Get FL training status |

### 8.1 Basic API Endpoints
Define core endpoints for health check, client registration, and model distribution.

In [ ]:
app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({'status': 'healthy', 'round': fl_state['current_round']})

@app.route('/register', methods=['POST'])
def register():
    data = request.json
    client_id = data.get('client_id')
    data_size = data.get('data_size', 0)
    fl_state['registered_clients'][client_id] = {'data_size': data_size}
    print(f'Client {client_id} registered with {data_size} samples')
    return jsonify({'status': 'registered', 'current_round': fl_state['current_round']})

@app.route('/model/download')
def download_model():
    if not os.path.exists(GLOBAL_MODEL_PATH):
        return jsonify({'error': 'No model available'}), 404
    client_id = request.args.get('client_id', 'unknown')
    print(f'Client {client_id} downloading model (round {fl_state["current_round"]})')
    return send_file(GLOBAL_MODEL_PATH, mimetype='application/octet-stream')

@app.route('/status')
def status():
    return jsonify({
        'current_round': fl_state['current_round'],
        'registered_clients': list(fl_state['registered_clients'].keys()),
        'weights_received_count': len(fl_state['received_weights']),
        'expected_clients': fl_state['expected_clients'],
        'best_bacc': fl_state['best_bacc'],
        'best_round': fl_state['best_round']
    })

@app.route('/model/upload_chunk', methods=['POST'])
def upload_chunk():
    data = request.json
    upload_id = data.get('upload_id')
    chunk_idx = data.get('chunk_idx')
    chunk_data = data.get('chunk_data')
    total_chunks = data.get('total_chunks')

    if upload_id not in chunked_uploads:
        chunked_uploads[upload_id] = {
            'client_id': data.get('client_id'),
            'total_chunks': total_chunks,
            'chunks': {},
            'data_size': data.get('data_size', 0),
            'round': data.get('round', 0)
        }
    chunked_uploads[upload_id]['chunks'][chunk_idx] = chunk_data
    return jsonify({'status': 'chunk_received', 'chunk_idx': chunk_idx})

@app.route('/model/upload_complete', methods=['POST'])
def upload_complete():
    data = request.json
    upload_id = data.get('upload_id')
    client_id = data.get('client_id')
    data_size = data.get('data_size', 0)

    if upload_id not in chunked_uploads:
        return jsonify({'error': 'Upload not found'}), 404

    upload_data = chunked_uploads[upload_id]
    total_chunks = upload_data['total_chunks']

    # Reassemble
    encoded_data = ''.join(upload_data['chunks'][i] for i in range(total_chunks))
    compressed_data = base64.b64decode(encoded_data)
    weights_data = gzip.decompress(compressed_data)
    weights = torch.load(io.BytesIO(weights_data), map_location='cpu')

    del chunked_uploads[upload_id]

    fl_state['received_weights'][client_id] = weights
    fl_state['client_data_sizes'][client_id] = data_size

    num_received = len(fl_state['received_weights'])
    print(f'Weights received from {client_id} ({num_received}/{fl_state["expected_clients"]})')

    # Auto aggregate if all clients uploaded
    result = None
    if num_received >= fl_state['expected_clients']:
        result = do_aggregation()

    response = {'status': 'received', 'total_received': num_received}
    if result:
        response['aggregation'] = result
        response['new_round'] = fl_state['current_round']
    return jsonify(response)

def do_aggregation():
    '''Perform FedAvg aggregation'''
    if fl_state['is_aggregating']:
        return None
    fl_state['is_aggregating'] = True

    try:
        weights_list = list(fl_state['received_weights'].values())
        data_sizes = [fl_state['client_data_sizes'][cid] for cid in fl_state['received_weights']]
        client_ids = list(fl_state['received_weights'].keys())

        print('='*50)
        print(f'FEDAVG AGGREGATION - Round {fl_state["current_round"]}')
        print(f'Clients: {client_ids}')
        print(f'Data sizes: {data_sizes}')

        aggregated = fedavg(weights_list, data_sizes)
        save_global_model(aggregated)

        # Evaluate
        eval_result = evaluate_model(aggregated)
        bacc = eval_result['balanced_accuracy']
        print(f'Test Accuracy: {eval_result["accuracy"]*100:.2f}%')
        print(f'Test Balanced Accuracy: {bacc*100:.2f}%')

        if bacc > fl_state['best_bacc']:
            fl_state['best_bacc'] = bacc
            fl_state['best_round'] = fl_state['current_round']
            torch.save(aggregated, BEST_MODEL_PATH)
            print(f'NEW BEST MODEL! BACC: {bacc*100:.2f}%')

        fl_state['evaluation_history'].append({
            'round': fl_state['current_round'],
            'bacc': bacc
        })

        old_round = fl_state['current_round']
        fl_state['current_round'] += 1
        fl_state['received_weights'] = {}
        fl_state['client_data_sizes'] = {}

        print(f'Round {old_round} -> {fl_state["current_round"]}')
        print('='*50)

        return {'status': 'aggregated', 'bacc': bacc}
    finally:
        fl_state['is_aggregating'] = False

print('Endpoints defined!')

Endpoints defined!


# 9. Start the FL Server

Run this cell to start the server. Once running:
1. **Copy the public URL**
2. Open `client_1.ipynb` and `client_2.ipynb` in separate tabs
3. Paste the URL in each client's configuration
4. Run the client notebooks to begin federated training

> **Note:** The server will keep running until you manually stop it or restart the runtime.

In [ ]:
print('='*50)
print('STARTING FL SERVER')
print('='*50)
print('')
print('>>> COPY THIS URL TO CLIENT NOTEBOOK! <<<')
print(f'Server URL: {SERVER_URL}')
print(f'Expected clients: {fl_state["expected_clients"]}')
print('')
print('After the server running, please run:')
print('  1. client_1.ipynb')
print('  2. client_2.ipynb')
print('='*50)

# Run Flask (use_reloader=False for Jupyter)
app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

STARTING FL SERVER

>>> COPY THIS URL TO CLIENT NOTEBOOK! <<<
Server URL: https://lily-snowplow-moodiness.ngrok-free.dev
Expected clients: 2

After the server running, please run:
  1. client_1.ipynb
  2. client_2.ipynb
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:23:44] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:23:44] "POST /register HTTP/1.1" 200 -


Client RSGM-YARSI registered with 425 samples


INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:23:44] "GET /model/download?client_id=RSGM-YARSI HTTP/1.1" 200 -


Client RSGM-YARSI downloading model (round 0)


...

INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:39:32] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:39:37] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:39:42] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:39:48] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:39:53] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:39:58] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:40:03] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:40:09] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:40:14] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:40:19] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:40:25] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:40:30] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026

Weights received from PathGen (2/2)
FEDAVG AGGREGATION - Round 2
Clients: ['RSGM-YARSI', 'PathGen']
Data sizes: [425, 595]
Global model saved to models/global_model.pth
Loaded pretrained weights for efficientnet-b0


INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:08] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:13] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:18] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:24] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:29] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:34] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:39] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:45] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:48] "POST /model/upload_complete HTTP/1.1" 200 -


Test Accuracy: 29.20%
Test Balanced Accuracy: 29.20%
Round 2 -> 3


INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 18:41:50] "GET /status HTTP/1.1" 200 -
